In [ ]:
# === Environment Detection & Path Config ===
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Robustly find the directory containing sales.csv
    DATA_DIR = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sales.csv' in files:
            DATA_DIR = root + '/'
            break
    if DATA_DIR is None:
        raise FileNotFoundError('Could not find sales.csv under /kaggle/input')
    OUT_DIR = '/kaggle/working/'
else:
    DATA_DIR = '../data/raw/'
    OUT_DIR = '../output/'

TRAIN_FILE = os.path.join(DATA_DIR, 'sales.csv')
TEST_FILE  = os.path.join(DATA_DIR, 'sample_submission.csv')
OUT_FILE   = os.path.join(OUT_DIR, 'submission.csv')

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data dir:    {DATA_DIR}")
print(f"Train file:  {TRAIN_FILE}")
print(f"Output file: {OUT_FILE}")

# Sales Forecasting — Baseline
**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01.

**Strategy:** Seasonal profile (month, day) + YoY geometric-mean growth.

## 1 — Load & Inspect Data

In [ ]:
train = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
test  = pd.read_csv(TEST_FILE,  parse_dates=['Date'])

print('Train shape:', train.shape)
print('Train date range:', train['Date'].min().date(), '→', train['Date'].max().date())
print()
print('Test shape:', test.shape)
print('Test date range:', test['Date'].min().date(), '→', test['Date'].max().date())
print()
train.tail()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(train['Date'], train['Revenue'], lw=0.7)
axes[0].set_title('Historical Daily Revenue'); axes[0].set_ylabel('Revenue')
axes[1].plot(train['Date'], train['COGS'], lw=0.7, color='orange')
axes[1].set_title('Historical Daily COGS'); axes[1].set_ylabel('COGS')
plt.tight_layout()
plt.show()

## 2 — Feature Engineering

In [ ]:
train['year']  = train['Date'].dt.year
train['month'] = train['Date'].dt.month
train['day']   = train['Date'].dt.day

annual = train.groupby('year')[['Revenue', 'COGS']].sum()
print('Annual totals:')
print(annual)

In [ ]:
# YoY growth rate (geometric mean, 2013-2022)
full_years = annual.loc[2013:2022]
yoy_rev  = full_years['Revenue'].pct_change().dropna()
yoy_cogs = full_years['COGS'].pct_change().dropna()
growth_rev  = (1 + yoy_rev).prod() ** (1 / len(yoy_rev))
growth_cogs = (1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))

print(f'YoY Revenue growth: {growth_rev:.4f}  ({(growth_rev-1)*100:.2f}%/yr)')
print(f'YoY COGS    growth: {growth_cogs:.4f}  ({(growth_cogs-1)*100:.2f}%/yr)')

## 3 — Seasonal Profile

In [ ]:
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')
train['rev_norm']  = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS']    / annual_means['COGS']

seasonal = (
    train
    .groupby(['month', 'day'])[['rev_norm', 'cogs_norm']]
    .mean()
    .reset_index()
)
print('Seasonal profile rows:', len(seasonal))

## 4 — Predict Test Period

In [ ]:
base_rev  = annual.loc[2022, 'Revenue']  / 365
base_cogs = annual.loc[2022, 'COGS']     / 365

test = test.copy()
test['month'] = test['Date'].dt.month
test['day']   = test['Date'].dt.day
test['year']  = test['Date'].dt.year
test['years_ahead'] = test['year'] - 2022

test = test.merge(seasonal, on=['month', 'day'], how='left')
test['rev_norm']  = test['rev_norm'].fillna(1.0)
test['cogs_norm'] = test['cogs_norm'].fillna(1.0)

test['Revenue'] = (base_rev  * growth_rev**test['years_ahead']  * test['rev_norm']).round(2)
test['COGS']    = (base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']).round(2)

print('Predictions sample:')
test[['Date','Revenue','COGS']].head(10)

## 5 — Export Submission

In [ ]:
submission = test[['Date', 'Revenue', 'COGS']].copy()
submission['Date'] = pd.to_datetime(submission['Date']).dt.strftime('%Y-%m-%d')

os.makedirs(OUT_DIR, exist_ok=True)
submission.to_csv(OUT_FILE, index=False)

print(f'Saved {len(submission)} rows to {OUT_FILE}')
submission.head()